[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH06/06_04_begin/notebooks/06_04_score_challenge.ipynb)

# 06_04 Challenge: Score your model against the baseline

Turn the evaluation outputs into a scorecard, then decide whether the fine-tuned model is ready to ship.

This notebook doesn't need a GPU. The default CPU runtime works fine.

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH06")

import ridgeline

## Load the evaluation outputs

In [ ]:
# Outputs from 06_01. Use the course copy, or upload your own and change the path.
OUTPUTS_FILE = ridgeline.DATA / "eval_outputs.jsonl"

outputs = ridgeline.load_tickets(OUTPUTS_FILE)
tickets = {t["id"]: t for t in ridgeline.load_eval_tickets()}
print(f"{len(outputs)} tickets with a baseline and a fine-tuned reply")

## Your task

**1. Write `score_model(outputs, tickets, model_name)`.** For one model, `"baseline"` or `"finetuned"`, return a dictionary with these keys, each a fraction from 0 to 1:

- `triage_accuracy`: both Category and Escalate correct
- `header_format`: the reply has valid Category and Escalate lines
- `voice_pass`: the reply passes every voice rule
- `escalation_recall`: of the tickets that should be escalated, the share the model escalated
- `by_category`: a dictionary of triage accuracy for each of the six categories

**2. Write `ready_to_ship(baseline, finetuned)`.** Return `True` only if the fine-tuned model meets all four ship criteria:

- Its triage accuracy is higher than the baseline's.
- Its header format is at least 95 percent.
- Its voice pass rate is at least 80 percent.
- Its escalation recall is 100 percent. Missing an injury or a chargeback threat is never acceptable.

In [ ]:
def score_model(outputs, tickets, model_name):
    # TODO: return triage_accuracy, header_format, voice_pass, escalation_recall, and by_category
    return {}


def ready_to_ship(baseline, finetuned):
    # TODO: return True only if the fine-tuned model meets all four ship criteria
    return False

## Build the scorecard

In [ ]:
import pandas as pd

baseline = score_model(outputs, tickets, "baseline")
finetuned = score_model(outputs, tickets, "finetuned")

headline = ["triage_accuracy", "header_format", "voice_pass", "escalation_recall"]
display(pd.DataFrame({"Baseline": {k: baseline.get(k) for k in headline},
                      "Fine-tuned": {k: finetuned.get(k) for k in headline}}).style.format("{:.0%}", na_rep="-"))
display(pd.DataFrame({"Baseline": baseline.get("by_category", {}),
                      "Fine-tuned": finetuned.get("by_category", {})}).style.format("{:.0%}", na_rep="-"))
print("Ready to ship:", ready_to_ship(baseline, finetuned))

## Check your work

In [ ]:
# Check your work. Each line prints PASS or FAIL.

def check(label, passed):
    print(f"{'PASS' if passed else 'FAIL'}  {label}")

keys = {"triage_accuracy", "header_format", "voice_pass", "escalation_recall", "by_category"}
check("score_model returns all five keys", keys <= set(finetuned))
check("Every rate is between 0 and 1", keys <= set(finetuned) and all(0 <= finetuned[k] <= 1 for k in keys - {"by_category"}))
check("by_category covers all six categories", set(finetuned.get("by_category", {})) == set(ridgeline.CATEGORIES))

escalations = [o for o in outputs if tickets[o["id"]]["escalate"] == "Yes"]
expected = sum(ridgeline.check_reply(tickets[o["id"]], o["finetuned"])["escalate_ok"] for o in escalations) / len(escalations)
check("escalation_recall counts only tickets that should be escalated", abs(finetuned.get("escalation_recall", -1) - expected) < 1e-9)

perfect = dict(finetuned, triage_accuracy=1.0, header_format=1.0, voice_pass=1.0, escalation_recall=1.0)
missed_one = dict(perfect, escalation_recall=0.9)
check("ready_to_ship passes a model that meets every criterion", ready_to_ship(dict(perfect, triage_accuracy=0.5), perfect) is True)
check("ready_to_ship fails a model that missed an escalation", ready_to_ship(dict(perfect, triage_accuracy=0.5), missed_one) is False)